# ITT consolidado + IPM - Barrio Obrero (oficial, 45 manzanas) (Cali)

Indice de Transformacion Territorial (**ITT**) con las 5 dimensiones reales
ponderadas, cruzado por manzana, replicando el estandar de Ciudad Paraiso, y
superpuesto con el **IPM** (indice de pobreza multidimensional) de cada manzana.

| Dimension | Peso | Variables | Direccion |
|---|---|---|---|
| Seguridad | 0.27 | hurtos + homicidios | inversa |
| Movilidad | 0.22 | siniestros viales | inversa |
| Desarrollo Social | 0.19 | VIF + rinas + SPA | inversa |
| Entorno Urbano | 0.17 | censo arboreo | directa |
| Desarrollo Economico | 0.15 | negocios + empleo | directa |

`ITT = SUMA(dimension x peso)`. Normalizacion relativa a la zona. 4 niveles:
Activacion 0-40 (#D55E00), Consolidacion 40-60 (#E69F00), Transformacion 60-80
(#009E73), Escala 80-100 (#0072B2). **Clic en una manzana**: compara su ITT
(transformacion, mayor=mejor) con su IPM (pobreza, mayor=peor) y las 15 privaciones.

Salida: `mapa_html/Barrio_obrero/Barrio_obrero_actual/mapa_itt_barrio_obrero.html`.

## 0. Setup Colab (opcional)

In [ ]:
def _en_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False
if _en_colab():
    import os,subprocess,sys
    subprocess.run([sys.executable,'-m','pip','install','-q','geopandas','folium','mapclassify','rasterio','pillow'],check=False)
    print('Colab: dependencias instaladas')
else:
    print('Entorno local (uv).')


## 1. Entorno, rutas y paleta

In [ ]:
import os, json
import numpy as np, pandas as pd, geopandas as gpd
import folium
# Paleta Okabe-Ito
OKI_AZUL="#0072B2"; OKI_NARANJA="#E69F00"; OKI_VERDE="#009E73"; OKI_AMARILLO="#F0E442"
OKI_AZUL_C="#56B4E9"; OKI_BERMELL="#D55E00"; OKI_MORADO="#CC79A7"; OKI_GRIS="#333333"
BG="#F4F6F9"; LIMITE="#1B4F8A"; CRS_WEB=4326; CRS_M=3116
def _ruta(*cands):
    for p in cands:
        if os.path.isfile(p) or os.path.isdir(p): return os.path.abspath(p)
    return None
def _bo(*parts):
    return _ruta(os.path.join("..","data","Barrio_Obrero_geojson",*parts),
                 os.path.join("data","Barrio_Obrero_geojson",*parts),
                 os.path.join("/content","data","Barrio_Obrero_geojson",*parts))
def _dd(*parts):
    return _ruta(os.path.join("..","data",*parts),
                 os.path.join("data",*parts),
                 os.path.join("/content","data",*parts))
ZONA_DIR="barrio_obrero_completo"
DATADIR=_bo(ZONA_DIR)
IPM_PATH=_bo("IPM_Barrio_Obrero_completo.geojson")
POLY_PATH=_bo("Geojson_Barrio_Obrero.geojson")
LIMITE_PATH=_bo(ZONA_DIR,"00_limite_barrio_obrero_completo.geojson")
CAI_PATH=_bo(ZONA_DIR,"5_Entorno_Urbano","CAI_MECAL_CALI_OBRERO.geojson")
NDVI_DIR=_bo(ZONA_DIR,"5_Entorno_Urbano","NDVI_Barrio_Obrero","NDVI_Barrio_Obrero")
DEFICIT_PATH=_dd("5_Dimension_Entorno_Urbano","sec_vivienda","DEF_HAB_SISBEN.shp")
if DATADIR is None or POLY_PATH is None:
    raise FileNotFoundError("Faltan datos base de Barrio Obrero")
def _en(d,pref):
    if d is None: return None
    for f in os.listdir(d):
        if f.startswith(pref) and f.endswith(".geojson"): return os.path.join(d,f)
    return None
POLIGONO_ZONA=gpd.read_file(POLY_PATH).to_crs(CRS_WEB)
_zg=POLIGONO_ZONA.geometry.union_all()
_c=_zg.centroid
_b=POLIGONO_ZONA.total_bounds
BO_BOUNDS=[[float(_b[1]),float(_b[0])],[float(_b[3]),float(_b[2])]]
# salida
OUT_DIR=None
for _d in [os.path.join("..","data_processed_geo"),"data_processed_geo","/content/data_processed_geo"]:
    try: os.makedirs(_d,exist_ok=True); OUT_DIR=os.path.abspath(_d); break
    except Exception: pass
_raiz=os.path.dirname(OUT_DIR.rstrip(os.sep))
MAPAS_DIR=os.path.abspath(os.path.join(_raiz,"mapa_html","Barrio_obrero","Barrio_obrero_actual"))
os.makedirs(MAPAS_DIR,exist_ok=True)
ZOOM=15
print("Zona:",len(POLIGONO_ZONA),"poligono | centro",round(_c.y,5),round(_c.x,5))
print("Salida:",MAPAS_DIR)

PESOS={'Seguridad':0.27,'Movilidad':0.22,'Desarrollo Social':0.19,'Entorno Urbano':0.17,'Desarrollo Economico':0.15}
assert abs(sum(PESOS.values())-1.0)<1e-6
ITT_NIVELES=[(0.0,40.0,'Activacion','#D55E00'),(40.0,60.0,'Consolidacion','#E69F00'),
             (60.0,80.0,'Transformacion','#009E73'),(80.0,100.0,'Escala','#0072B2')]
def _itt_nivel(v):
    v=float(v)
    for lo,hi,nom,col in ITT_NIVELES:
        if v<=hi: return nom,col
    return ITT_NIVELES[-1][2],ITT_NIVELES[-1][3]


## 2. Calculo del ITT por manzana + IPM y privaciones

In [ ]:
PRIV=["ANALF_","BAJO_","INFANCIA_","INASIS_","REZAGO_","TRAB_INFAN","DEPEN_","INFOR_",
      "SALUD_","ASEGU_","HACI_","PARED_","EXCRE_","PISOS_","AGUA_"]
PRIV_LABEL={"ANALF_":"Analfabetismo","BAJO_":"Bajo logro educativo","INFANCIA_":"Trabajo infancia",
    "INASIS_":"Inasistencia escolar","REZAGO_":"Rezago escolar","TRAB_INFAN":"Trabajo infantil",
    "DEPEN_":"Dependencia economica","INFOR_":"Empleo informal","SALUD_":"Barreras salud",
    "ASEGU_":"Sin aseguramiento","HACI_":"Hacinamiento","PARED_":"Material paredes",
    "EXCRE_":"Sin excretas","PISOS_":"Pisos inadecuados","AGUA_":"Sin agua"}
mz=gpd.read_file(IPM_PATH).to_crs(CRS_M)
mz=mz[mz.geometry.notna()].reset_index(drop=True)
mz["mid"]=range(len(mz)); mz["area_m2"]=mz.geometry.area.values
mz["ipm"]=pd.to_numeric(mz["ipm"],errors="coerce").fillna(0.0)
for c in PRIV:
    mz[c]=pd.to_numeric(mz[c],errors="coerce").fillna(0.0) if c in mz.columns else 0.0
def norm_inv(s):
    s=s.astype(float); rng=s.max()-s.min()
    return pd.Series(1.0,index=s.index) if rng==0 else 1.0-(s-s.min())/rng
def norm_dir(s):
    s=s.astype(float); rng=s.max()-s.min()
    return pd.Series(0.5,index=s.index) if rng==0 else (s-s.min())/rng
EVT={"homic":"02_seguridad_homicidios","hurtos":"03_seguridad_hurtos",
     "rinas":"04_convivencia_rinas","spa":"05_convivencia_spa","vif":"06_cohesion_vif",
     "siniestros":"07_movilidad_siniestros","arboles":"08_entorno_censo_arboreo",
     "negocios":"09_desarrollo_economico_negocios"}
FILTRO_AGR={"rinas":"RIÑAS","spa":"SUSTANCIAS PSICOACTIVAS"}
C={}
for k,pref in EVT.items():
    p=_en(DATADIR,pref)
    if p is None: C[k]=pd.Series(0,index=range(len(mz))); continue
    g=gpd.read_file(p).to_crs(CRS_M)
    g=g[g.geometry.notna() & g.geom_type.str.contains("Point")]
    if k in FILTRO_AGR and "agrupado" in g.columns:
        g=g[g["agrupado"].astype(str).str.strip().str.upper()==FILTRO_AGR[k]]
    if len(g)==0: C[k]=pd.Series(0,index=range(len(mz))); continue
    j=gpd.sjoin(g[["geometry"]],mz[["mid","geometry"]],predicate="within",how="inner")
    C[k]=j.groupby("mid").size().reindex(range(len(mz)),fill_value=0)
C=pd.DataFrame(C)
# empleo de negocios
pneg=_en(DATADIR,"09_desarrollo_economico_negocios")
if pneg:
    gneg=gpd.read_file(pneg).to_crs(CRS_M)
    gneg["_emp"]=pd.to_numeric(gneg["personal_o"],errors="coerce").fillna(0) if "personal_o" in gneg.columns else 0
    jn=gpd.sjoin(gneg[["_emp","geometry"]],mz[["mid","geometry"]],predicate="within",how="inner")
    C["empleo"]=jn.groupby("mid")["_emp"].sum().reindex(range(len(mz)),fill_value=0)
else:
    C["empleo"]=0
mz["d_seguridad"]=(norm_inv(C["hurtos"]+C["homic"])*100).round(1).values
mz["d_movilidad"]=(norm_inv(C["siniestros"])*100).round(1).values
mz["d_dessocial"]=(norm_inv(C["vif"]+C["rinas"]+C["spa"])*100).round(1).values
mz["d_entorno"]=(norm_dir(C["arboles"])*100).round(1).values
mz["d_economico"]=(norm_dir(C["negocios"]+C["empleo"])*100).round(1).values
for k in C.columns: mz["n_"+k]=C[k].values
DIMS=[("Seguridad","d_seguridad"),("Movilidad","d_movilidad"),("Desarrollo Social","d_dessocial"),
      ("Entorno Urbano","d_entorno"),("Desarrollo Economico","d_economico")]
PCOL={n:c for n,c in DIMS}
mz["itt"]=sum(mz[PCOL[k]]*w for k,w in PESOS.items()).round(1)
ITT_ZONA=round(float(mz["itt"].mean()),1)
IPM_ZONA=round(float(mz["ipm"].mean()),1)
DIM_ZONA={nom:round(float(mz[col].mean()),1) for nom,col in DIMS}
_nz,_=_itt_nivel(ITT_ZONA)
mz=mz.to_crs(CRS_WEB)
print("Manzanas:",len(mz),"| ITT zona: %.1f -> %s | IPM zona: %.1f"%(ITT_ZONA,_nz,IPM_ZONA))
for nom,col in DIMS: print("  %-22s %.1f  (peso %.2f)"%(nom,DIM_ZONA[nom],PESOS[nom]))

## 3. Tabla resumen ITT por dimension

In [ ]:
_tab=pd.DataFrame({"Dimension":[n for n,_ in DIMS]+["ITT CONSOLIDADO"],
  "Peso":[PESOS[n] for n,_ in DIMS]+[1.0],
  "Score (0-100)":[DIM_ZONA[n] for n,_ in DIMS]+[ITT_ZONA],
  "Aporte":[round(DIM_ZONA[n]*PESOS[n],1) for n,_ in DIMS]+[ITT_ZONA],
  "Nivel":[_itt_nivel(DIM_ZONA[n])[0] for n,_ in DIMS]+[_itt_nivel(ITT_ZONA)[0]]})
print(_tab.to_string(index=False))

## 4. Exportacion de lamina (deck) - helper reutilizable

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#seg-panel, #eu-panel, #ipm-panel, #itt-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#seg-panel, #eu-panel, #ipm-panel, #itt-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#seg-panel, #eu-panel, #ipm-panel, #itt-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      var mp = window[MAP_ID];
      // 2) fitBounds al poligono. maxZoom ENTERO evita el zoom fraccional que
      //    descoloca las capas en html2canvas (desfase al imprimir).
      var capturado = false;
      function trasEncuadre(){
        if(capturado){ return; }
        capturado = true;
        try { if(mp){ mp.invalidateSize(false); } } catch(e){}
        // 3) Margen corto tras asentar panes/tiles, luego capturar.
        setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
        }, 350);
      }
      // Disparar fitBounds y esperar 'moveend'; fallback por timeout si no llega.
      try {
        if(mp){
          mp.once('moveend', trasEncuadre);
          mp.fitBounds(ZONA_BOUNDS, {padding:[10,10], maxZoom: 17, animate: false});
          setTimeout(trasEncuadre, 1200);
        } else {
          trasEncuadre();
        }
      } catch(e){ trasEncuadre(); }
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

## 5. Mapa base (estandar Ciudad Paraiso)

In [ ]:
import folium
from folium.plugins import HeatMap
m=folium.Map(location=[_c.y,_c.x],zoom_start=ZOOM,tiles=None,control_scale=True,
             zoom_snap=0.25,zoom_delta=0.1,max_zoom=22)
m.get_root().width='100%'; m.get_root().height='100%'
m.get_root().header.add_child(folium.Element(
    "<style>html,body{height:100%;margin:0;padding:0}"
    ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;width:100%;height:100%}"
    ".leaflet-top,.leaflet-bottom{z-index:1000}"
    "#itt-panel{z-index:1001;background:rgba(255,255,255,.95);backdrop-filter:blur(4px);"
    "-webkit-backdrop-filter:blur(4px);box-shadow:0 4px 15px rgba(0,0,0,.2)}"
    "#heat-intensity-ctrl{z-index:1001}</style>"))
folium.TileLayer(tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',attr='Google',name='Google Streets',show=True,max_zoom=22,max_native_zoom=20).add_to(m)
folium.TileLayer(tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',attr='Esri',name='Esri Satelite',show=False,max_zoom=22,max_native_zoom=20).add_to(m)
folium.TileLayer(tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',attr='Google',name='Google Satelite',show=False,max_zoom=22,max_native_zoom=20).add_to(m)
_mundo=[[-180,-85],[180,-85],[180,85],[-180,85],[-180,-85]]; _ag=[]
for _gm in (list(_zg.geoms) if _zg.geom_type=='MultiPolygon' else [_zg]):
    _ag.append([[x,y] for x,y in _gm.exterior.coords])
folium.GeoJson({'type':'Feature','properties':{},'geometry':{'type':'Polygon','coordinates':[_mundo]+_ag}},
    name='Mascara (foco en la zona)',control=False,
    style_function=lambda f:{'fillColor':'#000','color':'#000','weight':0,'fillOpacity':0.18}).add_to(m)


## 6. Capas: ITT por manzana (clicable), IPM por manzana, eventos

In [ ]:
import branca.colormap as _bcm
# Payload por manzana para la interaccion JS (ITT vs IPM + dimensiones + privaciones)
MZ_DATA=[]
for _,r in mz.iterrows():
    MZ_DATA.append({"id":int(r["mid"]),"itt":float(r["itt"]),"ipm":float(r["ipm"]),
        "dims":{n:float(r[c]) for n,c in DIMS},
        "ev":{"hurtos":int(r["n_hurtos"]),"homic":int(r["n_homic"]),"siniestros":int(r["n_siniestros"]),
              "vif":int(r["n_vif"]),"rinas":int(r["n_rinas"]),"spa":int(r["n_spa"]),
              "arboles":int(r["n_arboles"]),"negocios":int(r["n_negocios"]),"empleo":int(r["n_empleo"])},
        "priv":{PRIV_LABEL[p]:float(r[p]) for p in PRIV}})
# Coropleta ITT (clicable) - capa activa
_fg=folium.FeatureGroup(name="ITT: manzanas (indice consolidado)",show=True)
for _,r in mz.iterrows():
    nom,col=_itt_nivel(float(r["itt"]))
    _gj=folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=(lambda f,c=col:{"fillColor":c,"color":"#ffffff","weight":0.6,"fillOpacity":0.82}),
        highlight_function=lambda f:{"weight":2.5,"color":"#111","fillOpacity":0.9},
        tooltip="ITT %.1f (%s) - clic para comparar con IPM"%(float(r["itt"]),nom))
    _gj.add_child(folium.Popup("mz:%d"%int(r["mid"]),show=False))
    _gj.add_to(_fg)
_fg.add_to(m)
# Coropleta IPM (clicable) - capa apagada; mayor IPM = mas pobreza (cividis)
_ipmmax=max(float(mz["ipm"].max()),1e-6)
_cmap=_bcm.linear.viridis.scale(0,_ipmmax)
_fgi=folium.FeatureGroup(name="IPM: manzanas (pobreza multidimensional)",show=False)
for _,r in mz.iterrows():
    _gj=folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=(lambda f,c=_cmap(float(r["ipm"])):{"fillColor":c,"color":"#ffffff","weight":0.6,"fillOpacity":0.82}),
        highlight_function=lambda f:{"weight":2.5,"color":"#111","fillOpacity":0.9},
        tooltip="IPM %.1f - clic para comparar con ITT"%float(r["ipm"]))
    _gj.add_child(folium.Popup("mz:%d"%int(r["mid"]),show=False))
    _gj.add_to(_fgi)
_fgi.add_to(m)
# Dimensiones individuales (apagadas)
for nomd,cold in DIMS:
    fgd=folium.FeatureGroup(name="Dimension: "+nomd,show=False)
    for _,r in mz.iterrows():
        _n,_col=_itt_nivel(float(r[cold]))
        folium.GeoJson(r["geometry"].__geo_interface__,
            style_function=(lambda f,c=_col:{"fillColor":c,"color":"#fff","weight":0.4,"fillOpacity":0.72}),
            tooltip="%s: %.0f"%(nomd,float(r[cold]))).add_to(fgd)
    fgd.add_to(m)
# Eventos puntuales (apagados)
_EVT_CAPAS=[("Seguridad: hurtos","03_seguridad_hurtos","#0072B2",None),
            ("Seguridad: homicidios","02_seguridad_homicidios","#D55E00",None),
            ("Desarrollo Social: rinas","04_convivencia_rinas","#009E73","RIÑAS"),
            ("Desarrollo Social: SPA","05_convivencia_spa","#E69F00","SUSTANCIAS PSICOACTIVAS"),
            ("Movilidad: siniestros","07_movilidad_siniestros","#56B4E9",None),
            ("Entorno: arbolado","08_entorno_censo_arboreo","#009E73",None),
            ("Desarrollo eco: negocios","09_desarrollo_economico_negocios","#0072B2",None)]
for nomc,pref,colr,filt in _EVT_CAPAS:
    pth=_en(DATADIR,pref)
    if pth is None: continue
    gg=gpd.read_file(pth).to_crs(CRS_WEB)
    gg=gg[gg.geometry.notna() & gg.geom_type.str.contains("Point")]
    if filt is not None and "agrupado" in gg.columns:
        gg=gg[gg["agrupado"].astype(str).str.strip().str.upper()==filt]
    fgp=folium.FeatureGroup(name=nomc+" ("+str(len(gg))+")",show=False)
    for _,rr in gg.iterrows():
        folium.CircleMarker([rr.geometry.y,rr.geometry.x],radius=3,color=colr,fill=True,
            fill_color=colr,fill_opacity=0.75,weight=0.5).add_to(fgp)
    fgp.add_to(m)
# VIF: heatmap agregado (privacidad)
_pv=_en(DATADIR,"06_cohesion_vif")
if _pv:
    _gv=gpd.read_file(_pv).to_crs(CRS_WEB)
    _gv=_gv[_gv.geometry.notna() & _gv.geom_type.str.contains("Point")]
    from folium.plugins import HeatMap as _HM
    _fgv=folium.FeatureGroup(name="Desarrollo Social: VIF (calor, agregado)",show=False)
    _HM([[p.y,p.x] for p in _gv.geometry],gradient={0.0:"#f2f0f7",0.5:"#9e9ac8",1.0:"#54278f"},
        radius=20,blur=16,min_opacity=0.35).add_to(_fgv)
    _fgv.add_to(m)
# red peatonal contexto
_prp=_en(DATADIR,"11_red_peatonal_osm")
if _prp:
    gr=gpd.read_file(_prp).to_crs(CRS_WEB)
    folium.GeoJson(gr.to_json(),name="Movilidad: red peatonal OSM (contexto)",show=False,
        style_function=lambda f:{"color":"#888","weight":1.2,"opacity":0.7}).add_to(m)
print("Capas ITT/IPM agregadas.")

## 7. Limite, estacion CAI, leyenda y controles

In [ ]:
folium.GeoJson(POLIGONO_ZONA.to_json(),name='Limite Barrio Obrero',show=True,
    style_function=lambda f:{'color':LIMITE,'weight':2.5,'fill':False}).add_to(m)
# Estacion de policia (CAI/MECAL) - gestion
if CAI_PATH:
    _cai=gpd.read_file(CAI_PATH).to_crs(CRS_WEB)
    _fgc=folium.FeatureGroup(name='Gestion: estacion policia (CAI)',show=True)
    for _,_r in _cai.iterrows():
        _sig=str(_r.get('SIGLA',_r.get('UNIDAD','CAI')))
        folium.Marker([_r.geometry.y,_r.geometry.x],
            tooltip='Estacion: '+_sig,
            icon=folium.Icon(color='darkblue',icon='shield',prefix='fa')).add_to(_fgc)
    _fgc.add_to(m)

_items=""
for lo,hi,nom,col in ITT_NIVELES:
    _items+=('<div style="display:flex;align-items:center;gap:6px;margin:1px 0">'
             '<i style="width:16px;height:12px;background:%s;display:inline-block;border:1px solid #999"></i>'
             '<span style="font-size:10px">%s (%d-%d)</span></div>'%(col,nom,int(lo),int(hi)))
_ley=('<div style="position:fixed;bottom:24px;right:24px;z-index:1000;background:'+BG+';'
      'padding:8px 10px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);font-family:system-ui,Segoe UI,sans-serif;color:'+OKI_GRIS+';max-width:220px">'
      '<div style="font-weight:700;font-size:11px;margin-bottom:4px">Nivel ITT</div>'+_items+
      '<div style="font-size:9px;color:#888;margin-top:4px">ITT = suma ponderada de 5 dimensiones. Clic en manzana: ITT vs IPM.</div></div>')
m.get_root().html.add_child(folium.Element(_ley))
m.fit_bounds(BO_BOUNDS,padding=(30,30))
folium.LayerControl(collapsed=True,position='topright').add_to(m)
print('Mapa construido.')


## 8. Panel flotante: ITT consolidado + comparacion clic-manzana (ITT vs IPM)

In [ ]:
_dim_labels=[nom+" ("+("%.2f"%PESOS[nom])+")" for nom,_ in DIMS]
_dim_vals=[DIM_ZONA[nom] for nom,_ in DIMS]
_cont={nom:0 for _lo,_hi,nom,_c in ITT_NIVELES}
for _v in mz["itt"]:
    _n,_=_itt_nivel(_v); _cont[_n]+=1
_payload=json.dumps({"mz":MZ_DATA,"itt_zona":ITT_ZONA,"ipm_zona":IPM_ZONA,
    "dim_labels":_dim_labels,"dim_vals":_dim_vals,
    "niveles":[[lo,hi,nom,col] for lo,hi,nom,col in ITT_NIVELES]},ensure_ascii=False)
_resumen=('<div style="margin:2px 0 6px 0">ITT de zona: <b style="font-size:15px;color:%s">%.1f</b> / 100 &rarr; <b>%s</b> &nbsp;|&nbsp; IPM zona: <b>%.1f</b></div>'%(LIMITE,ITT_ZONA,_nz,IPM_ZONA))
_resumen+='<div style="margin:4px 0 2px;font-weight:700;font-size:11px">Manzanas por nivel ITT:</div>'
for _lo,_hi,nom,col in ITT_NIVELES:
    _resumen+=('<div style="display:flex;align-items:center;gap:6px;margin:1px 0;font-size:11px">'
               '<i style="width:12px;height:12px;background:%s;display:inline-block;border-radius:2px;border:1px solid #777"></i>'
               '<span>%s: <b>%d</b></span></div>'%(col,nom,_cont[nom]))
_panel=r'''
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<div id="itt-panel" style="position:fixed;bottom:25px;left:20px;z-index:1001;max-width:420px;width:min(420px,45vw);max-height:86vh;overflow-y:auto;padding:11px 13px;border-radius:10px;font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="display:flex;align-items:center;justify-content:space-between;gap:8px">
    <div style="font-weight:700;font-size:13px">ITT vs IPM - Barrio Obrero (oficial)</div>
    <button id="itt-toggle" title="Minimizar / expandir" style="cursor:pointer;border:1px solid #bbb;background:#fff;color:#333;width:24px;height:22px;border-radius:5px;font-size:13px;line-height:1;font-weight:700;flex:0 0 auto">_</button>
  </div>
  <div id="itt-body">
    <div style="font-size:10px;color:#666;margin:4px 0 6px 0">Indice de Transformacion Territorial (mayor=mejor) comparado con el IPM de pobreza (mayor=peor). Clic en una manzana.</div>
    __RESUMEN__
    <div id="itt-sel" style="margin-top:8px;padding:8px;border:1px dashed #bbb;border-radius:8px;background:rgba(244,246,249,.6)">
      <div style="font-size:11px;color:#666">Selecciona una manzana en el mapa para comparar su <b>ITT</b> y su <b>IPM</b>.</div>
    </div>
    <details open style="margin-top:8px">
      <summary style="cursor:pointer;font-weight:600;color:__LIM__;outline:none">Dimensiones del ITT (zona o manzana) / Ocultar</summary>
      <div style="position:relative;height:210px;width:100%;margin-top:6px"><canvas id="itt-chart"></canvas></div>
      <div style="font-size:9px;color:#777;margin-top:3px">Score 0-100 por dimension; el peso aparece junto al nombre.</div>
    </details>
    <details style="margin-top:6px">
      <summary style="cursor:pointer;font-weight:600;color:#777;outline:none">Privaciones IPM de la manzana / Ver</summary>
      <div style="position:relative;height:230px;width:100%;margin-top:6px"><canvas id="itt-priv-chart"></canvas></div>
    </details>
  </div>
</div>
<script>
(function(){
  var P=__PAYLOAD__;
  var MZ={}; P.mz.forEach(function(d){ MZ[d.id]=d; });
  function colItt(v){ return v<40?"#D55E00":(v<60?"#E69F00":(v<80?"#009E73":"#0072B2")); }
  function nivel(v){ for(var i=0;i<P.niveles.length;i++){ if(v<=P.niveles[i][1]) return P.niveles[i][2]; } return P.niveles[P.niveles.length-1][2]; }
  var chartDim=null, chartPriv=null;
  function dibujarDim(labels,vals,titulo){
    var cv=document.getElementById("itt-chart"); if(!cv||typeof Chart==="undefined") return;
    if(chartDim) chartDim.destroy();
    chartDim=new Chart(cv.getContext("2d"),{type:"bar",
      data:{labels:labels,datasets:[{label:titulo,data:vals,backgroundColor:vals.map(colItt),borderWidth:1}]},
      options:{indexAxis:"y",responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},title:{display:true,text:titulo,font:{size:10}}},
        scales:{x:{beginAtZero:true,max:100,ticks:{font:{size:9}},title:{display:true,text:"Score (0-100)",font:{size:9}}},y:{ticks:{font:{size:8}}}}}});
  }
  function dibujarPriv(d){
    var cv=document.getElementById("itt-priv-chart"); if(!cv||typeof Chart==="undefined") return;
    var labels=Object.keys(d.priv), vals=labels.map(function(k){return d.priv[k];});
    if(chartPriv) chartPriv.destroy();
    chartPriv=new Chart(cv.getContext("2d"),{type:"bar",
      data:{labels:labels,datasets:[{label:"% privacion",data:vals,backgroundColor:"#CC79A7",borderWidth:1}]},
      options:{indexAxis:"y",responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},
        scales:{x:{beginAtZero:true,max:100,ticks:{font:{size:8}},title:{display:true,text:"% de hogares con la privacion",font:{size:8}}},y:{ticks:{font:{size:7}}}}}});
  }
  function lecturaZona(){
    document.getElementById("itt-sel").innerHTML=
      '<div style="font-size:11px;color:#666">Mostrando promedio de zona. <b>Clic en una manzana</b> para el detalle ITT vs IPM.</div>';
    dibujarDim(P.dim_labels,P.dim_vals,"ITT por dimension (zona)");
  }
  function seleccionar(id){
    var d=MZ[id]; if(!d) return;
    var niv=nivel(d.itt);
    var html='<div style="font-weight:700;font-size:12px;margin-bottom:4px">Manzana '+id+'</div>';
    html+='<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px">';
    html+='<div style="padding:6px;border-radius:6px;background:#fff;border-left:4px solid '+colItt(d.itt)+'"><div style="font-size:10px;color:#777">ITT (transformacion)</div><div style="font-size:18px;font-weight:700;color:'+colItt(d.itt)+'">'+d.itt.toFixed(1)+'</div><div style="font-size:10px">'+niv+'</div></div>';
    html+='<div style="padding:6px;border-radius:6px;background:#fff;border-left:4px solid #6a51a3"><div style="font-size:10px;color:#777">IPM (pobreza)</div><div style="font-size:18px;font-weight:700;color:#54278f">'+d.ipm.toFixed(1)+'</div><div style="font-size:10px">'+(d.ipm>=33.3?"pobreza alta":"bajo umbral")+'</div></div>';
    html+='</div>';
    var dItt=d.itt-P.itt_zona, dIpm=d.ipm-P.ipm_zona;
    html+='<div style="font-size:10px;color:#555;margin-top:6px">vs zona: ITT '+(dItt>=0?"+":"")+dItt.toFixed(1)+' ('+(dItt>=0?"mejor":"peor")+'), IPM '+(dIpm>=0?"+":"")+dIpm.toFixed(1)+' ('+(dIpm<=0?"mejor":"peor")+').</div>';
    document.getElementById("itt-sel").innerHTML=html;
    var labels=P.dim_labels, vals=P.dim_labels.map(function(l){ var nm=l.replace(/ \(.*/,""); return d.dims[nm]; });
    dibujarDim(labels,vals,"ITT por dimension (manzana "+id+")");
    dibujarPriv(d);
  }
  function initColapso(){
    var b=document.getElementById("itt-toggle"),bo=document.getElementById("itt-body");
    if(b&&bo) b.addEventListener("click",function(){var o=(bo.style.display==="none");bo.style.display=o?"":"none";b.textContent=o?"_":"+";});
  }
  function hookMapa(){
    var mp=window["__MVAR__"];
    if(!mp||!mp.on){ setTimeout(hookMapa,300); return; }
    mp.on("popupopen",function(e){
      try{
        var c=e.popup.getContent();
        if(typeof c==="string" && c.indexOf("mz:")===0){
          var id=parseInt(c.slice(3),10);
          e.popup.remove();
          seleccionar(id);
        }
      }catch(err){}
    });
  }
  function start(){ setTimeout(function(){ lecturaZona(); },350); initColapso(); hookMapa(); }
  if(document.readyState==="complete"){ start(); }
  else{ window.addEventListener("load",start); }
})();
</script>
'''
_panel=(_panel.replace("__GRIS__",OKI_GRIS).replace("__LIM__",LIMITE)
             .replace("__RESUMEN__",_resumen).replace("__PAYLOAD__",_payload)
             .replace("__MVAR__","m"))
m.get_root().html.add_child(folium.Element(_panel))
print("Panel ITT vs IPM inyectado.")

## 9. Exportacion del HTML

In [ ]:
agregar_export(m, 'ITT consolidado + IPM - Barrio Obrero (oficial, 45 manzanas)', 'itt_bo')
html=os.path.join(MAPAS_DIR,'mapa_itt_barrio_obrero.html')
m.save(html)
print('Mapa ITT consolidado exportado:', html)
m